# Notebook 8 — Vector Databases & RAG
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Embeddings vectoriels & similarite](#embeddings)
2. [Indexation : HNSW, IVF, PQ](#index)
3. [Vector Databases](#vectordb)
4. [BM25 & recherche hybride](#bm25)
5. [RAG (Retrieval-Augmented Generation)](#rag)
6. [Re-ranking & Cross-Encoder](#rerank)
7. [Evaluation RAG](#eval)
8. [Questions d’entretien](#questions)


---
## 1. Embeddings vectoriels & similarité <a name='embeddings'></a>

### Pourquoi des embeddings ?
Représenter du texte comme un vecteur dense pour pouvoir comparer sémantiquement des documents.

### Modèles d’embedding courants
| Modele | Dim | Tokens max | Score MTEB |
|---|---|---|---|
| text-embedding-ada-002 (OpenAI) | 1536 | 8191 | 60.9 |
| text-embedding-3-large (OpenAI) | 3072 | 8191 | 64.6 |
| E5-large-v2 | 1024 | 512 | 62.3 |
| BGE-large-en-v1.5 | 1024 | 512 | 63.6 |
| sentence-transformers/all-MiniLM | 384 | 256 | 56.3 |
| Mistral-embed | 1024 | 8192 | 65.0 |

### Similarités vectorielles
- **Cosine** : `cos(u,v) = u.v / (|u| * |v|)` — standard pour les embeddings de texte
- **Produit scalaire (IP)** : `u.v` — plus rapide, equivalent si vecteurs normalises
- **Distance L2 (Euclidienne)** : `||u-v||` — adaptee aux embeddings d'images parfois


In [ ]:
import numpy as np

# ============================================================
# EMBEDDINGS ET SIMILARITES
# ============================================================
def cosine_similarity(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-10)

def inner_product(u, v):
    return np.dot(u, v)

def l2_distance(u, v):
    return np.linalg.norm(u - v)

def batch_cosine(query, corpus):
    """Cosine similarity vectorisee : query (d,) vs corpus (N, d)"""
    q_norm = query / (np.linalg.norm(query) + 1e-10)
    c_norm = corpus / (np.linalg.norm(corpus, axis=1, keepdims=True) + 1e-10)
    return c_norm @ q_norm

np.random.seed(42)
d = 128  # dimension des embeddings

# Simuler des embeddings semantiquement proches ou eloignes
emb_chat_banque   = np.random.randn(d);   emb_chat_banque   /= np.linalg.norm(emb_chat_banque)
emb_compte_banque = emb_chat_banque + np.random.randn(d) * 0.2  # tres proche
emb_compte_banque /= np.linalg.norm(emb_compte_banque)
emb_cuisine       = np.random.randn(d);   emb_cuisine       /= np.linalg.norm(emb_cuisine)
emb_code_python   = np.random.randn(d);   emb_code_python   /= np.linalg.norm(emb_code_python)

corpus = np.stack([emb_chat_banque, emb_compte_banque, emb_cuisine, emb_code_python])
labels = ["chatbot bancaire", "compte bancaire", "recette de cuisine", "code Python"]

query = emb_chat_banque + np.random.randn(d) * 0.1
query /= np.linalg.norm(query)

scores = batch_cosine(query, corpus)
ranked = np.argsort(scores)[::-1]

print("=== Recherche par similarite cosine ===")
print(f"Query: proche de 'chatbot bancaire'")
print(f"{'Rang':>4}  {'Document':25s}  {'Cosine':>8}  {'L2 dist':>8}")
for rank, idx in enumerate(ranked):
    l2 = l2_distance(query, corpus[idx])
    print(f"  {rank+1}    {labels[idx]:25s}  {scores[idx]:>8.4f}  {l2:>8.4f}")

print("\n=== Proprietes des metriques ===")
print("  Cosine   : insensible a la norme, standard NLP")
print("  IP       : equivalent cosine si L2-normalise (plus rapide)")
print("  L2       : sensible a la norme, utilisee en vision parfois")

# Normalisation
print("\n=== L2-normalisation : cosine == IP ===")
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])
u_n = u / np.linalg.norm(u)
v_n = v / np.linalg.norm(v)
print(f"  Cosine(u,v)           = {cosine_similarity(u, v):.6f}")
print(f"  IP(u_norm, v_norm)    = {inner_product(u_n, v_n):.6f}")
print(f"  Identiques : {np.isclose(cosine_similarity(u,v), inner_product(u_n,v_n))}")


---
## 2. Indexation : HNSW, IVF, PQ <a name='index'></a>

### Probleme : Approximate Nearest Neighbor (ANN)
Chercher le plus proche voisin parmi N vecteurs en dimension d.
- **Brute force** : O(N * d) — trop lent pour N > 10^6
- **ANN** : trouver le voisin *approximativement* le plus proche en O(log N)

### HNSW (Hierarchical Navigable Small World)
Structure de graphe hierarchique :
- **Couche 0** : tous les vecteurs, connexions locales
- **Couche k** : sous-ensemble aleatoire, connexions longue distance
- **Construction** : chaque noeud se connecte aux M voisins les plus proches
- **Recherche** : partir de la couche haute, descendre layer by layer, beam search

Hyperparametres :
- `M` : nb de connexions par noeud (16-64 typique). Plus grand => meilleur recall, plus de memoire
- `ef_construction` : taille du beam pendant la construction (100-200)
- `ef_search` : taille du beam pendant la recherche (controle precision/vitesse)

### IVF (Inverted File Index)
- Clusteriser les vecteurs en K clusters (K-means)
- Au query : trouver les `nprobe` clusters les plus proches, bruteforce dedans
- Rapide en construction, bon compromis vitesse/precision

### PQ (Product Quantization)
- Diviser le vecteur en M sous-vecteurs
- Quantifier chaque sous-vecteur independamment (256 centroides)
- Compression : d floats -> M bytes (ex: 1024d -> 32 bytes = 32x compression!)

### Compromis recall/vitesse
| Methode | Recall@10 | QPS | Memoire |
|---|---|---|---|
| Brute force | 100% | Faible | O(N*d*4 bytes) |
| IVF-flat | 95-99% | Moyen | O(N*d*4 bytes) |
| HNSW | 97-99% | Eleve | O(N*d*4 + graph) |
| IVF-PQ | 85-95% | Tres eleve | Tres compresse |


In [ ]:
import numpy as np
from collections import defaultdict

# ============================================================
# BRUTE FORCE KNN -- baseline
# ============================================================
def brute_force_knn(query, corpus, k=5, metric="cosine"):
    if metric == "cosine":
        norms = np.linalg.norm(corpus, axis=1, keepdims=True)
        corpus_n = corpus / (norms + 1e-10)
        q_n = query / (np.linalg.norm(query) + 1e-10)
        scores = corpus_n @ q_n
        return np.argsort(scores)[::-1][:k], np.sort(scores)[::-1][:k]
    elif metric == "l2":
        dists = np.linalg.norm(corpus - query, axis=1)
        idx = np.argsort(dists)[:k]
        return idx, dists[idx]

# ============================================================
# HNSW SIMPLIFIE -- principes algorithmiques
# ============================================================
class SimpleHNSW:
    """
    Implementation simplifiee de HNSW (2 niveaux seulement pour la demo)
    Montre les principes : graphe navigable + hierarchie de couches
    """
    def __init__(self, M=8, ef=20, seed=42):
        self.M  = M    # nb de connexions par noeud
        self.ef = ef   # taille du beam de recherche
        self.rng = np.random.RandomState(seed)
        self.data   = []
        self.graphs = [defaultdict(set), defaultdict(set)]   # 2 couches
        self.levels = []

    def _assign_level(self):
        """Niveau aleatoire : 1 avec proba 1/M, 0 sinon"""
        return 1 if self.rng.random() < 1.0 / self.M else 0

    def _cosine(self, a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def _greedy_search(self, query, entry, layer, k):
        """Recherche greedy dans une couche : beam search"""
        visited = {entry}
        candidates = [(self._cosine(query, self.data[entry]), entry)]
        best_k = list(candidates)

        while candidates:
            _, curr = max(candidates)
            candidates.remove((_, curr))
            neighbors = self.graphs[layer][curr]
            improved = False
            for nb in neighbors:
                if nb not in visited:
                    visited.add(nb)
                    sim = self._cosine(query, self.data[nb])
                    candidates.append((sim, nb))
                    best_k.append((sim, nb))
                    improved = True
            if not improved and len(best_k) >= k:
                break

        best_k.sort(reverse=True)
        return [idx for _, idx in best_k[:k]]

    def add(self, vector):
        idx = len(self.data)
        self.data.append(vector)
        level = self._assign_level()
        self.levels.append(level)

        if idx == 0:
            return

        # Connexions dans les couches
        for layer in range(min(level + 1, 2)):
            # Trouver les M voisins les plus proches deja inseres
            existing = [i for i, l in enumerate(self.levels) if l >= layer and i != idx]
            if not existing:
                continue
            sims = [(self._cosine(vector, self.data[i]), i) for i in existing]
            sims.sort(reverse=True)
            neighbors = [i for _, i in sims[:self.M]]
            for nb in neighbors:
                self.graphs[layer][idx].add(nb)
                self.graphs[layer][nb].add(idx)

    def search(self, query, k=5):
        if not self.data:
            return []
        # Partir du plus haut niveau et descendre
        entry = 0
        for layer in range(1, -1, -1):
            entry_candidates = self._greedy_search(query, entry, layer, k=1)
            if entry_candidates:
                entry = entry_candidates[0]
        return self._greedy_search(query, entry, 0, k=k)

np.random.seed(42)
N, d = 500, 32
corpus = np.random.randn(N, d)
corpus_n = corpus / np.linalg.norm(corpus, axis=1, keepdims=True)

# Construire l index HNSW
hnsw = SimpleHNSW(M=8, ef=20)
for v in corpus_n:
    hnsw.add(v)

print("=== HNSW vs Brute Force ===")
k = 5
n_queries = 50
hits = 0
for _ in range(n_queries):
    q = np.random.randn(d)
    q = q / np.linalg.norm(q)
    true_nn, _ = brute_force_knn(q, corpus_n, k=k)
    hnsw_nn    = hnsw.search(q, k=k)
    hits += len(set(true_nn) & set(hnsw_nn))

recall = hits / (n_queries * k)
print(f"Recall@{k} sur {n_queries} queries : {recall:.2%}")
print(f"Connexions moy par noeud (couche 0): {np.mean([len(v) for v in hnsw.graphs[0].values()]):.1f}")
print(f"Noeuds en couche 1 : {sum(1 for l in hnsw.levels if l >= 1)} / {N}")


In [ ]:
import numpy as np

# ============================================================
# IVF (Inverted File Index) -- from scratch
# ============================================================
class IVFIndex:
    """
    Inverted File Index:
    - K-means pour creer nlist clusters
    - Recherche dans les nprobe clusters les plus proches
    """
    def __init__(self, nlist=16, nprobe=4):
        self.nlist  = nlist
        self.nprobe = nprobe
        self.centroids = None
        self.inverted_lists = None

    def _kmeans(self, data, k, n_iter=20, seed=42):
        rng = np.random.RandomState(seed)
        centroids = data[rng.choice(len(data), k, replace=False)]
        for _ in range(n_iter):
            dists = np.linalg.norm(data[:, np.newaxis] - centroids, axis=2)
            labels = np.argmin(dists, axis=1)
            new_centroids = np.array([
                data[labels == c].mean(axis=0) if (labels==c).any() else centroids[c]
                for c in range(k)
            ])
            if np.allclose(centroids, new_centroids):
                break
            centroids = new_centroids
        return centroids, labels

    def build(self, data):
        self.centroids, labels = self._kmeans(data, self.nlist)
        self.data   = data
        self.labels = labels
        self.inverted_lists = {c: np.where(labels == c)[0] for c in range(self.nlist)}
        return self

    def search(self, query, k=5):
        # Trouver les nprobe clusters les plus proches
        dists = np.linalg.norm(self.centroids - query, axis=1)
        top_clusters = np.argsort(dists)[:self.nprobe]
        # Bruteforce dans ces clusters
        candidates = np.concatenate([self.inverted_lists[c] for c in top_clusters])
        if len(candidates) == 0:
            return []
        q_n = query / (np.linalg.norm(query) + 1e-10)
        cand_data = self.data[candidates]
        cand_n = cand_data / (np.linalg.norm(cand_data, axis=1, keepdims=True) + 1e-10)
        scores = cand_n @ q_n
        top = np.argsort(scores)[::-1][:k]
        return list(candidates[top])

np.random.seed(42)
N, d = 2000, 64
corpus = np.random.randn(N, d)
corpus_n = corpus / np.linalg.norm(corpus, axis=1, keepdims=True)

ivf = IVFIndex(nlist=32, nprobe=4).build(corpus_n)

k, n_q = 5, 100
hits_ivf = 0
for _ in range(n_q):
    q = np.random.randn(d); q /= np.linalg.norm(q)
    true_nn, _ = brute_force_knn(q, corpus_n, k=k)
    ivf_nn     = ivf.search(q, k=k)
    hits_ivf  += len(set(true_nn) & set(ivf_nn))

print("=== IVF Index ===")
print(f"N={N}, d={d}, nlist={ivf.nlist}, nprobe={ivf.nprobe}")
print(f"Recall@{k} sur {n_q} queries : {hits_ivf/(n_q*k):.2%}")
avg_cands = np.mean([len(ivf.inverted_lists[c]) for c in range(ivf.nlist)])
print(f"Vecteurs par cluster (moy)  : {avg_cands:.0f}")
print(f"Vecteurs explores par query : {avg_cands * ivf.nprobe:.0f} / {N} ({avg_cands*ivf.nprobe/N:.0%})")

# ============================================================
# PQ (Product Quantization) -- compression
# ============================================================
print("\n=== Product Quantization (PQ) ===")
def pq_compress(data, M=8, K=256, seed=42):
    """
    M sous-vecteurs, K centroides chacun
    Compression: d floats (4B each) -> M uint8 (1B each)
    """
    rng = np.random.RandomState(seed)
    N, d = data.shape
    d_sub = d // M
    codebooks = []
    codes = np.zeros((N, M), dtype=np.uint8)

    for m in range(M):
        sub = data[:, m*d_sub:(m+1)*d_sub]
        # K-means leger
        centroids = sub[rng.choice(N, K, replace=False)]
        for _ in range(5):
            dists = np.linalg.norm(sub[:, np.newaxis] - centroids, axis=2)
            labels = np.argmin(dists, axis=1)
            centroids = np.array([sub[labels==k].mean(0) if (labels==k).any()
                                   else centroids[k] for k in range(K)])
        codebooks.append(centroids)
        codes[:, m] = labels.astype(np.uint8)

    return codes, codebooks

d = 64
N_small = 500
data = np.random.randn(N_small, d).astype(np.float32)
M, K = 8, 256

codes, codebooks = pq_compress(data, M=M, K=K)

mem_original = N_small * d * 4
mem_pq       = N_small * M * 1
print(f"  Vecteurs: {N_small} x {d}D float32")
print(f"  Memoire originale : {mem_original:,} bytes = {mem_original/1024:.1f} KB")
print(f"  Memoire PQ ({M} sous-vect, {K} codes): {mem_pq:,} bytes = {mem_pq/1024:.1f} KB")
print(f"  Compression : {mem_original/mem_pq:.0f}x")
print(f"  Codes shape : {codes.shape}  (uint8 : 1 byte par sous-vecteur)")


---
## 3. Vector Databases <a name='vectordb'></a>

### Principales solutions
| Solution | Type | Index | Metadonnees | Use case |
|---|---|---|---|---|
| Faiss | Librairie | HNSW, IVF, PQ | Non | Recherche pure, custom |
| Chroma | DB embarquee | HNSW (hnswlib) | Oui | Dev, prototypage rapide |
| Weaviate | DB distribuee | HNSW | Oui | Production, cloud |
| Pinecone | SaaS manage | Propietaire | Oui | Production serverless |
| Qdrant | DB open-source | HNSW | Oui | Production self-hosted |
| pgvector | Extension PG | IVFFlat, HNSW | Oui (SQL) | Deja sur PostgreSQL |
| Milvus | DB distribuee | HNSW, IVF, PQ | Oui | Large scale |
| Redis (VSS) | In-memory | HNSW, FLAT | Oui | Cache + vecteurs |

### Metadonnees et filtrage hybride
Les VDB permettent de filtrer sur les metadonnees **avant** ou **apres** la recherche vectorielle :
```python
# Exemple Chroma / Qdrant
results = collection.query(
    query_embeddings=[query_emb],
    n_results=5,
    where={"source": "rapport_annuel_2024", "langue": "fr"}  # filtrage
)
```

### Architecture d’un VDB
```
Documents -> Chunking -> Embedding Model -> Vecteurs
                                              |
                                        [VDB Index]
                                              |
Query -> Embedding -> ANN Search -> Top-K docs -> LLM
```


In [ ]:
import numpy as np
from collections import defaultdict

# ============================================================
# VECTOR DATABASE MINIMALISTE -- from scratch
# ============================================================
class SimpleVectorDB:
    """
    VDB minimaliste avec:
    - Stockage de vecteurs + metadonnees
    - Index HNSW simplifie (ici brute force pour la demo)
    - Filtrage par metadonnees
    - CRUD basique
    """
    def __init__(self, dim, metric="cosine"):
        self.dim      = dim
        self.metric   = metric
        self.vectors  = {}   # id -> vecteur
        self.metadata = {}   # id -> dict metadonnees
        self._next_id = 0

    def add(self, vectors, metadata=None):
        """Ajouter des vecteurs avec metadonnees optionnelles"""
        if isinstance(vectors, np.ndarray) and vectors.ndim == 1:
            vectors = [vectors]
        if metadata is None:
            metadata = [{} for _ in vectors]

        ids = []
        for vec, meta in zip(vectors, metadata):
            doc_id = self._next_id
            self.vectors[doc_id]  = vec / (np.linalg.norm(vec) + 1e-10)
            self.metadata[doc_id] = meta
            self._next_id += 1
            ids.append(doc_id)
        return ids

    def delete(self, doc_id):
        self.vectors.pop(doc_id, None)
        self.metadata.pop(doc_id, None)

    def update(self, doc_id, vector=None, metadata=None):
        if vector is not None:
            self.vectors[doc_id] = vector / (np.linalg.norm(vector) + 1e-10)
        if metadata is not None:
            self.metadata[doc_id].update(metadata)

    def _filter(self, where):
        """Retourne les ids qui matchent le filtre"""
        if not where:
            return set(self.vectors.keys())
        return {
            doc_id for doc_id, meta in self.metadata.items()
            if all(meta.get(k) == v for k, v in where.items())
        }

    def search(self, query, k=5, where=None):
        """Recherche ANN avec filtrage optionnel"""
        candidates = self._filter(where)
        if not candidates:
            return []
        ids = list(candidates)
        mat = np.stack([self.vectors[i] for i in ids])
        q_n = query / (np.linalg.norm(query) + 1e-10)

        if self.metric == "cosine":
            scores = mat @ q_n
            top = np.argsort(scores)[::-1][:k]
        else:
            dists = np.linalg.norm(mat - q_n, axis=1)
            top = np.argsort(dists)[:k]
            scores = -dists

        return [(ids[i], float(scores[i]), self.metadata[ids[i]]) for i in top]

    def __len__(self):
        return len(self.vectors)

# Demo avec des documents BNP
np.random.seed(42)
d = 64

vdb = SimpleVectorDB(dim=d, metric="cosine")

documents = [
    ("Rapport annuel BNP Paribas 2024 - resultats financiers", {"source": "rapport_2024", "type": "finance", "lang": "fr"}),
    ("BNP Paribas annual report 2024 - financial results",     {"source": "rapport_2024", "type": "finance", "lang": "en"}),
    ("Conditions generales des comptes courants BNP",          {"source": "cgu", "type": "legal", "lang": "fr"}),
    ("API documentation for BNP payment services",             {"source": "api_doc", "type": "tech", "lang": "en"}),
    ("Politique de credit immobilier 2024",                    {"source": "credit_policy", "type": "finance", "lang": "fr"}),
    ("Machine learning pour la detection de fraude",           {"source": "ml_paper", "type": "tech", "lang": "fr"}),
    ("Fraud detection using deep learning at BNP",             {"source": "ml_paper", "type": "tech", "lang": "en"}),
    ("Taux d interet et politique monetaire BCE 2024",         {"source": "macro", "type": "finance", "lang": "fr"}),
]

embs = np.random.randn(len(documents), d)
for i, (_, meta) in enumerate(documents):
    # Simuler des embeddings semantiquement coherents
    if "finance" in meta["type"]:
        embs[i] += np.array([1.0] + [0.0]*(d-1)) * 2
    elif "tech" in meta["type"]:
        embs[i] += np.array([0.0, 1.0] + [0.0]*(d-2)) * 2

ids = vdb.add(embs, [m for _, m in documents])

print("=== Simple Vector Database ===")
print(f"Documents stockes : {len(vdb)}")

# Recherche sans filtre
query_emb = embs[0] + np.random.randn(d) * 0.2
print(f"\n1. Recherche libre (top 3):")
results = vdb.search(query_emb, k=3)
for doc_id, score, meta in results:
    print(f"   id={doc_id}  score={score:.4f}  meta={meta}")

# Recherche avec filtre metadonnees
print(f"\n2. Filtree: lang=fr, type=finance (top 3):")
results = vdb.search(query_emb, k=3, where={"lang": "fr", "type": "finance"})
for doc_id, score, meta in results:
    print(f"   id={doc_id}  score={score:.4f}  '{documents[doc_id][0][:50]}'")

# CRUD
print(f"\n3. Update + Delete:")
vdb.update(0, metadata={"reviewed": True})
vdb.delete(1)
print(f"   Apres delete(1) : {len(vdb)} documents")
print(f"   Metadata doc 0  : {vdb.metadata[0]}")


---
## 4. BM25 & recherche hybride <a name='bm25'></a>

### BM25 (Best Match 25)
Amelioration de TF-IDF, standard en recherche full-text depuis 25 ans.

```
BM25(d, q) = sum_t IDF(t) * TF_sat(t, d)

IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1)

TF_sat(t, d) = tf(t,d) * (k1 + 1) / (tf(t,d) + k1 * (1 - b + b * len(d)/avgdl))
```

- `k1` : saturation TF (1.2-2.0 typique) — a partir d'un seuil, repeter un mot n'apporte plus
- `b` : normalisation longueur (0.75 typique) — penalise les documents longs

### Avantages BM25 vs embeddings
| | BM25 | Embeddings |
|---|---|---|
| Type | Sparse (lexical) | Dense (semantique) |
| Matching | Exact (tokens) | Flou (sens) |
| OOV | Gere naturellement | Problematique |
| Acronymes, noms propres | Excellent | Mediocre |
| Synonymes, paraphrases | Mauvais | Excellent |
| Vitesse | Ultra-rapide | Lent (GPU) |

### Recherche hybride (Sparse + Dense)
Combiner BM25 et embedding pour le meilleur des deux mondes :
```
score_hybrid = alpha * score_bm25_norm + (1-alpha) * score_emb
```
Ou RRF (Reciprocal Rank Fusion) :
```
RRF(d) = sum_r 1 / (k + rank_r(d))   k=60 typique
```


In [ ]:
import numpy as np
from collections import Counter
import math

# ============================================================
# BM25 -- from scratch
# ============================================================
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b  = b
        self.corpus    = []
        self.idf       = {}
        self.avgdl     = 0
        self.doc_freqs = []

    def fit(self, corpus):
        """corpus : list of list of tokens"""
        self.corpus = corpus
        N = len(corpus)
        self.avgdl = np.mean([len(d) for d in corpus])

        # DF: nombre de docs contenant chaque terme
        df = Counter()
        for doc in corpus:
            df.update(set(doc))

        # IDF BM25 (formule Robertson)
        self.idf = {
            t: math.log((N - n + 0.5) / (n + 0.5) + 1)
            for t, n in df.items()
        }

        # TF par document
        self.doc_freqs = [Counter(doc) for doc in corpus]
        return self

    def score(self, query_tokens, doc_idx):
        doc  = self.corpus[doc_idx]
        tf   = self.doc_freqs[doc_idx]
        dl   = len(doc)
        total = 0.0
        for t in query_tokens:
            if t not in self.idf:
                continue
            tf_t = tf.get(t, 0)
            # Saturation TF + normalisation longueur
            tf_sat = tf_t * (self.k1 + 1) / (
                tf_t + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            )
            total += self.idf[t] * tf_sat
        return total

    def search(self, query_tokens, k=5):
        scores = [(i, self.score(query_tokens, i)) for i in range(len(self.corpus))]
        scores.sort(key=lambda x: -x[1])
        return scores[:k]

# Corpus de documents financiers
corpus_texts = [
    "BNP Paribas annonce des resultats financiers record pour 2024",
    "La banque centrale europeenne augmente les taux d interet",
    "Rapport annuel BNP Paribas 2024 resultats benefices",
    "Detection de fraude par apprentissage automatique deep learning",
    "Credit immobilier taux hypothecaire BNP Paribas",
    "Machine learning applications finance banque",
    "BNP Paribas acquisition fusion banque investissement",
    "Politique monetaire taux interet inflation zone euro",
]
tokenized = [text.lower().split() for text in corpus_texts]

bm25 = BM25(k1=1.5, b=0.75).fit(tokenized)

queries = [
    "resultats financiers BNP".lower().split(),
    "taux interet banque centrale".lower().split(),
    "fraude machine learning".lower().split(),
]

print("=== BM25 Search ===")
for query in queries:
    results = bm25.search(query, k=3)
    print(f"\nQuery: '{' '.join(query)}'")
    for rank, (idx, score) in enumerate(results):
        print(f"  {rank+1}. [{score:.4f}] {corpus_texts[idx][:60]}")


In [ ]:
import numpy as np
from collections import Counter
import math

# ============================================================
# RECHERCHE HYBRIDE : BM25 + Embeddings + RRF
# ============================================================
def reciprocal_rank_fusion(rankings, k=60):
    """
    Fusion de plusieurs rankings par RRF
    rankings : list de list de doc_ids (du mieux classe au moins bien)
    k : constante de lissage (60 par defaut)
    """
    scores = Counter()
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

def normalize_scores(scores_list):
    """Min-max normalisation des scores [0, 1]"""
    mn, mx = min(scores_list), max(scores_list)
    if mx == mn:
        return [0.5] * len(scores_list)
    return [(s - mn) / (mx - mn) for s in scores_list]

class HybridRetriever:
    def __init__(self, bm25_index, embeddings, alpha=0.5):
        self.bm25  = bm25_index
        self.embs  = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10)
        self.alpha = alpha

    def search_bm25(self, query_tokens, k):
        return [(idx, score) for idx, score in self.bm25.search(query_tokens, k)]

    def search_dense(self, query_emb, k):
        q = query_emb / (np.linalg.norm(query_emb) + 1e-10)
        scores = self.embs @ q
        top = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in top]

    def search_hybrid_linear(self, query_tokens, query_emb, k=5):
        """Combinaison lineaire apres normalisation"""
        bm25_raw  = self.search_bm25(query_tokens, k*3)
        dense_raw = self.search_dense(query_emb, k*3)

        bm25_norm  = dict(zip([i for i,_ in bm25_raw],  normalize_scores([s for _,s in bm25_raw])))
        dense_norm = dict(zip([i for i,_ in dense_raw], normalize_scores([s for _,s in dense_raw])))

        all_ids = set(bm25_norm) | set(dense_norm)
        hybrid  = {idx: self.alpha * bm25_norm.get(idx, 0) + (1-self.alpha) * dense_norm.get(idx, 0)
                   for idx in all_ids}
        return sorted(hybrid.items(), key=lambda x: -x[1])[:k]

    def search_rrf(self, query_tokens, query_emb, k=5):
        """RRF : independant des echelles, plus robuste"""
        bm25_rank  = [i for i,_ in self.search_bm25(query_tokens, k*3)]
        dense_rank = [i for i,_ in self.search_dense(query_emb, k*3)]
        return reciprocal_rank_fusion([bm25_rank, dense_rank])[:k]

# Simulation
np.random.seed(42)
n_docs, d = 8, 32
# BM25 deja construit au-dessus
embs = np.random.randn(n_docs, d)
# Simuler coherence semantique
for i in [0, 2, 6]:  # docs "BNP + finance"
    embs[i] += np.array([2.0] + [0.0]*(d-1))
for i in [3, 5]:     # docs "ML"
    embs[i] += np.array([0.0, 2.0] + [0.0]*(d-2))

retriever = HybridRetriever(bm25, embs, alpha=0.5)

query_tokens = "resultats BNP machine learning".lower().split()
query_emb    = embs[0] + np.random.randn(d) * 0.3   # proche du doc 0

print("=== Recherche Hybride ===")
print(f"Query: '{' '.join(query_tokens)}'\n")

bm25_only  = retriever.search_bm25(query_tokens, 4)
dense_only = retriever.search_dense(query_emb, 4)
hybrid     = retriever.search_hybrid_linear(query_tokens, query_emb, 4)
rrf        = retriever.search_rrf(query_tokens, query_emb, 4)

print(f"{'BM25 seul':20s} : {[corpus_texts[i][:30] for i,_ in bm25_only[:3]]}")
print(f"{'Dense seul':20s} : {[corpus_texts[i][:30] for i,_ in dense_only[:3]]}")
print(f"{'Hybride (alpha=0.5)':20s} : {[corpus_texts[i][:30] for i,_ in hybrid[:3]]}")
print(f"{'RRF':20s} : {[corpus_texts[i][:30] for i,_ in rrf[:3]]}")
print("\n=> Hybride : meilleur rappel sur les termes exacts (BM25) ET la semantique")
print("   RRF recommande : robuste aux echelles differentes")


---
## 5. RAG (Retrieval-Augmented Generation) <a name='rag'></a>

### Pipeline RAG de base
```
1. Indexation (offline)
   Documents -> Chunking -> Embedding -> VDB

2. Generation (online)
   Question -> Embedding -> VDB search -> Top-K chunks
           -> Prompt = [System] + [Context chunks] + [Question]
           -> LLM -> Reponse
```

### Strategies de chunking
| Strategie | Description | Avantages |
|---|---|---|
| Fixed size | Chunk de N tokens, overlap O | Simple, rapide |
| Sentence | Decouper par phrases | Coherent |
| Recursive | Hierarchique (para > phrase > mot) | LangChain standard |
| Semantic | Chunker selon la coherence semantique | Meilleur, plus lent |
| Document-aware | Respecter la structure (titres, sections) | Pour PDF/Word |

### Parametres cles
- **Chunk size** : 256-512 tokens pour l'embedding, 512-2048 pour le contexte LLM
- **Overlap** : 10-20% pour eviter de couper les contextes importants
- **Top-K** : 3-10 chunks recupéres
- **Context window** : ne pas depasser la limite du LLM

### Problemes et solutions
| Probleme | Solution |
|---|---|
| Chunks trop petits (manque contexte) | Augmenter chunk size, parent document retrieval |
| Chunks trop grands (bruit) | Reduire chunk size, reranker |
| Hallucination | RAG fusion, verificateur de grounding |
| Requete ambigue | HyDE, query expansion, multi-query |
| Contexte perdu dans le LLM | Lost in the middle, reordering |


In [ ]:
import numpy as np
import re

# ============================================================
# CHUNKING STRATEGIES -- from scratch
# ============================================================
def fixed_size_chunker(text, chunk_size=200, overlap=50):
    """Decoupage par taille fixe avec overlap"""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
        if i + overlap >= len(words):
            break
    return chunks

def sentence_chunker(text, max_sentences=3):
    """Decoupage par phrases"""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunks.append(" ".join(sentences[i:i+max_sentences]))
    return chunks

def recursive_chunker(text, max_chunk=300, separators=None):
    """Decoupage hierarchique (style LangChain)"""
    if separators is None:
        separators = ["

", "
", ". ", " ", ""]
    if len(text.split()) <= max_chunk:
        return [text]
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks, current = [], ""
            for part in parts:
                if len((current + sep + part).split()) <= max_chunk:
                    current = (current + sep + part).strip()
                else:
                    if current:
                        chunks.append(current)
                    current = part.strip()
            if current:
                chunks.append(current)
            return [c for c in chunks if c.strip()]
    return [text]

# Document de test
document = """
BNP Paribas est une banque internationale francaise. Elle est presente dans 68 pays
et emploie plus de 190 000 personnes. Son siege social est situe a Paris.

La banque propose des services aux particuliers, aux entreprises et aux institutions.
Elle est specialisee dans la banque de detail, la gestion d actifs et la banque d investissement.
En 2024, BNP Paribas a affiche un produit net bancaire de 47 milliards d euros.

La transformation digitale est une priorite strategique. La banque investit massivement
dans l intelligence artificielle et le machine learning. Ces technologies sont utilisees
pour la detection de fraude, le scoring de credit et le service client automatise.
""".strip()

print("=== Strategies de Chunking ===")
c_fixed = fixed_size_chunker(document, chunk_size=50, overlap=10)
c_sent  = sentence_chunker(document, max_sentences=2)
c_recur = recursive_chunker(document, max_chunk=60)

print(f"Fixed size (50w, ov=10) : {len(c_fixed)} chunks")
for i, c in enumerate(c_fixed):
    print(f"  [{i}] ({len(c.split())} mots) {c[:60]}...")
print(f"\nSentence (2 phrases)   : {len(c_sent)} chunks")
for i, c in enumerate(c_sent):
    print(f"  [{i}] {c[:70]}...")
print(f"\nRecursive (max=60w)    : {len(c_recur)} chunks")
for i, c in enumerate(c_recur):
    print(f"  [{i}] ({len(c.split())} mots) {c[:60]}...")


In [ ]:
import numpy as np
import re

# ============================================================
# PIPELINE RAG COMPLET -- simulation
# ============================================================
class RAGPipeline:
    """
    Pipeline RAG complet:
    - Indexation de documents (chunking + embedding + stockage)
    - Retrieval (embedding query + ANN search)
    - Generation (construction du prompt + appel LLM simule)
    """
    def __init__(self, embed_dim=32, chunk_size=50, overlap=10, top_k=3):
        self.embed_dim  = embed_dim
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self.top_k      = top_k
        self.chunks     = []
        self.embeddings = None
        self.chunk_metadata = []

    def _fake_embed(self, texts, seed=None):
        """Simule un modele d embedding"""
        rng = np.random.RandomState(seed or hash(str(texts)) % (2**31))
        embs = rng.randn(len(texts), self.embed_dim)
        return embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10)

    def _chunk_text(self, text, source=""):
        words = text.split()
        chunks, metadata = [], []
        i = 0
        while i < len(words):
            chunk_words = words[i:i+self.chunk_size]
            chunk_text  = " ".join(chunk_words)
            chunks.append(chunk_text)
            metadata.append({"source": source, "start_word": i, "n_words": len(chunk_words)})
            i += self.chunk_size - self.overlap
            if i + self.overlap >= len(words):
                break
        return chunks, metadata

    def index(self, documents):
        """Indexer une liste de (text, source)"""
        all_chunks, all_meta = [], []
        for text, source in documents:
            chunks, meta = self._chunk_text(text, source)
            all_chunks.extend(chunks)
            all_meta.extend(meta)

        self.chunks          = all_chunks
        self.chunk_metadata  = all_meta
        self.embeddings      = self._fake_embed(all_chunks, seed=42)
        print(f"Indexe : {len(documents)} docs -> {len(all_chunks)} chunks")

    def retrieve(self, query, top_k=None):
        """Recuperer les top-k chunks les plus pertinents"""
        top_k = top_k or self.top_k
        q_emb = self._fake_embed([query], seed=hash(query) % (2**31))[0]
        scores = self.embeddings @ q_emb
        top = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], self.chunk_metadata[i], float(scores[i])) for i in top]

    def build_prompt(self, query, retrieved_chunks, system_prompt=None):
        """Construire le prompt RAG"""
        if system_prompt is None:
            system_prompt = "Tu es un assistant IA specialise en finance bancaire. Reponds en te basant uniquement sur le contexte fourni."
        context = "

".join([
            f"[Source: {meta['source']}, extrait {i+1}]
{chunk}"
            for i, (chunk, meta, _) in enumerate(retrieved_chunks)
        ])
        return f"""[SYSTEM]
{system_prompt}

[CONTEXT]
{context}

[QUESTION]
{query}

[REPONSE]"""

# Documents BNP
documents = [
    ("BNP Paribas est une banque internationale francaise presente dans 68 pays. "
     "Elle emploie 190000 personnes et son siege est a Paris. "
     "En 2024 le produit net bancaire atteint 47 milliards d euros.", "rapport_2024"),
    ("BNP utilise le machine learning pour la detection de fraude en temps reel. "
     "Les algorithmes analysent des milliers de transactions par seconde. "
     "La precision du systeme de detection atteint 99.2 pourcent.", "ml_paper"),
    ("Les taux d interet pratiques par BNP pour les credits immobiliers varient de 3.5 a 4.2 pourcent. "
     "La duree maximale est de 25 ans. Un apport de 10 pourcent est recommande.", "credit_guide"),
]

rag = RAGPipeline(embed_dim=32, chunk_size=30, overlap=5, top_k=3)
rag.index(documents)

print("\n=== RAG Pipeline ===")
query = "Quel est le produit net bancaire de BNP en 2024 ?"
retrieved = rag.retrieve(query)
prompt = rag.build_prompt(query, retrieved)

print(f"\nQuery: '{query}'")
print(f"\nChunks recuperes ({len(retrieved)}) :")
for chunk, meta, score in retrieved:
    print(f"  score={score:.4f}  source={meta['source']}  '{chunk[:60]}...'")
print(f"\nPrompt RAG (extrait) :")
print(prompt[:500] + "...")


---
## 6. Re-ranking & Cross-Encoder <a name='rerank'></a>

### Probleme du bi-encoder (embedding classique)
- Encoder la query et chaque document **independamment**
- Rapide mais approximatif : les interactions query-document sont perdues

### Cross-Encoder
- Encoder la paire `(query, document)` **ensemble**
- `score = model("[CLS] query [SEP] document [SEP]")` -> scalar
- Beaucoup plus precis car le modele voit les interactions
- Mais : N fois plus lent (doit processer chaque paire)

### Pipeline a deux etapes (standard en RAG avance)
```
1. Retrieval rapide (bi-encoder)  : top-100 candidats
2. Re-ranking (cross-encoder)     : top-5 finaux
3. Generation (LLM)               : reponse avec top-5
```

### Modeles cross-encoder populaires
- `cross-encoder/ms-marco-MiniLM-L-6-v2` : ultra-rapide, passage ranking
- `cross-encoder/ms-marco-electra-base` : meilleur, plus lent
- `BAAI/bge-reranker-large` : tres bon multilingue

### Quand utiliser le reranker ?
- Quand la precision > la vitesse (applications finance, legal, medical)
- Quand les queries sont complexes (plusieurs contraintes)
- Quand le domaine est specialise (les embeddings generaux sont moins bons)


In [ ]:
import numpy as np

# ============================================================
# CROSS-ENCODER SIMPLIFIE -- simulation
# ============================================================
class FakeCrossEncoder:
    """
    Simule un cross-encoder:
    - Prend (query, document) en entree
    - Retourne un score de pertinence
    Dans la realite: BERT/RoBERTa fine-tune sur MS-MARCO ou similaire
    """
    def __init__(self, d=32, seed=42):
        np.random.seed(seed)
        self.W_q = np.random.randn(d, d) * 0.1
        self.W_d = np.random.randn(d, d) * 0.1
        self.W_cross = np.random.randn(d, d) * 0.1
        self.v = np.random.randn(d) * 0.1

    def _fake_encode(self, text, seed=None):
        rng = np.random.RandomState(seed or abs(hash(text)) % (2**31))
        e = rng.randn(32)
        return e / (np.linalg.norm(e) + 1e-10)

    def predict(self, pairs):
        """
        pairs: list de (query, document)
        Retourne scores de pertinence
        """
        scores = []
        for query, doc in pairs:
            q_emb = self._fake_encode(query)
            d_emb = self._fake_encode(doc)
            # Interaction cross-encoder : concatenation + projection
            interaction = q_emb * d_emb   # element-wise product
            score = float(np.tanh(np.dot(self.v, interaction)))
            scores.append(score)
        return np.array(scores)

class TwoStageRetriever:
    """
    Retrieval en deux etapes:
    1. Bi-encoder : top-K rapide
    2. Cross-encoder : re-rank les K candidats
    """
    def __init__(self, documents, embed_dim=32):
        self.documents = documents
        self.bi_encoder  = None
        self.cross_encoder = FakeCrossEncoder(d=embed_dim)
        self.embed_dim   = embed_dim
        rng = np.random.RandomState(42)
        self.doc_embeddings = rng.randn(len(documents), embed_dim)
        self.doc_embeddings /= (np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True) + 1e-10)

    def bi_encode(self, text):
        rng = np.random.RandomState(abs(hash(text)) % (2**31))
        e = rng.randn(self.embed_dim)
        return e / (np.linalg.norm(e) + 1e-10)

    def retrieve(self, query, initial_k=10, final_k=3):
        # Etape 1 : Bi-encoder retrieval
        q_emb = self.bi_encode(query)
        bi_scores = self.doc_embeddings @ q_emb
        top_k_idx = np.argsort(bi_scores)[::-1][:initial_k]

        # Etape 2 : Cross-encoder reranking
        pairs  = [(query, self.documents[i]) for i in top_k_idx]
        cross_scores = self.cross_encoder.predict(pairs)

        # Re-trier par cross-encoder score
        reranked = sorted(zip(top_k_idx, cross_scores), key=lambda x: -x[1])[:final_k]
        return {
            "bi_encoder_top": [(int(i), float(bi_scores[i])) for i in top_k_idx[:5]],
            "reranked":       [(int(i), float(s)) for i, s in reranked]
        }

docs = [
    "BNP Paribas resultats financiers 2024 benefice net",
    "Detection de fraude par machine learning BNP",
    "Taux credit immobilier BNP Paribas conditions",
    "Recrutement ingenieur IA BNP Paribas Paris",
    "API paiement BNP integration documentation",
    "BNP Paribas rapport annuel investisseurs 2024",
    "Formation certifications bancaires BNP",
    "Securite informatique cyber BNP protection donnees",
    "BNP wealth management gestion de patrimoine",
    "Open banking PSD2 BNP Paribas APIs",
]

retriever = TwoStageRetriever(docs, embed_dim=32)
query = "resultats financiers BNP 2024 benefices"
result = retriever.retrieve(query, initial_k=6, final_k=3)

print("=== Two-Stage Retrieval (Bi-Encoder + Cross-Encoder) ===")
print(f"Query: '{query}'\n")
print("Bi-encoder top-5 (ANN rapide) :")
for idx, score in result["bi_encoder_top"]:
    print(f"  [{score:+.4f}]  {docs[idx][:55]}")
print("\nCross-encoder re-ranked top-3 (lent, precis) :")
for idx, score in result["reranked"]:
    print(f"  [{score:+.4f}]  {docs[idx][:55]}")
print("\n=> Le cross-encoder peut changer le classement du bi-encoder")
print("   Il tient compte des interactions fines query-document")


---
## 7. Evaluation RAG <a name='eval'></a>

### Metriques de retrieval
- **Recall@K** : proportion des documents pertinents recuperes dans le top-K
- **Precision@K** : proportion de documents pertinents parmi les K recuperes
- **MRR (Mean Reciprocal Rank)** : `1/rank_premier_pertinent`
- **NDCG@K** : Normalized Discounted Cumulative Gain (tient compte de l'ordre)

### Metriques de generation RAG
| Metrique | Mesure | Outil |
|---|---|---|
| Faithfulness | La reponse est-elle grounded dans le contexte ? | RAGAS |
| Answer Relevancy | La reponse repond-elle a la question ? | RAGAS |
| Context Precision | Le contexte recupere est-il pertinent ? | RAGAS |
| Context Recall | Le contexte couvre-t-il la reponse ideale ? | RAGAS |
| EM / F1 | Pour QA extractif (exact match) | SQuAD eval |

### RAGAS (framework d’evaluation RAG)
```python
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas import evaluate

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision]
)
```


In [ ]:
import numpy as np

# ============================================================
# METRIQUES DE RETRIEVAL -- from scratch
# ============================================================
def precision_at_k(retrieved, relevant, k):
    """P@K : fraction de retrieved[:k] qui sont pertinents"""
    top_k = set(retrieved[:k])
    return len(top_k & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    """R@K : fraction des pertinents recuperes dans retrieved[:k]"""
    top_k = set(retrieved[:k])
    return len(top_k & set(relevant)) / len(relevant) if relevant else 0

def mrr(retrieved_lists, relevant_lists):
    """Mean Reciprocal Rank"""
    rr_scores = []
    for retrieved, relevant in zip(retrieved_lists, relevant_lists):
        rr = 0.0
        for rank, doc_id in enumerate(retrieved):
            if doc_id in relevant:
                rr = 1.0 / (rank + 1)
                break
        rr_scores.append(rr)
    return np.mean(rr_scores)

def dcg_at_k(retrieved, relevant, k):
    """Discounted Cumulative Gain"""
    dcg = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = 1 if doc in relevant else 0
        dcg += rel / np.log2(i + 2)   # +2 car log2(1) = 0
    return dcg

def ndcg_at_k(retrieved, relevant, k):
    """Normalized DCG : DCG / ideal_DCG"""
    ideal = sorted([1 if d in relevant else 0 for d in retrieved], reverse=True)
    idcg = sum(rel / np.log2(i+2) for i, rel in enumerate(ideal[:k]))
    return dcg_at_k(retrieved, relevant, k) / idcg if idcg > 0 else 0.0

# Exemple d evaluation
queries = [
    {"retrieved": [0, 3, 1, 7, 4, 2], "relevant": {0, 1, 2}},
    {"retrieved": [5, 2, 8, 1, 3, 0], "relevant": {1, 3}},
    {"retrieved": [2, 4, 6, 0, 1, 3], "relevant": {2, 1}},
]

print("=== Metriques de Retrieval ===")
print(f"{'Metrique':15s}  Q0      Q1      Q2     Moy")
for k in [3, 5]:
    p_scores = [precision_at_k(q["retrieved"], q["relevant"], k) for q in queries]
    r_scores = [recall_at_k(q["retrieved"], q["relevant"], k)    for q in queries]
    n_scores = [ndcg_at_k(q["retrieved"], q["relevant"], k)      for q in queries]
    print(f"P@{k}            {p_scores[0]:.4f}  {p_scores[1]:.4f}  {p_scores[2]:.4f}  {np.mean(p_scores):.4f}")
    print(f"R@{k}            {r_scores[0]:.4f}  {r_scores[1]:.4f}  {r_scores[2]:.4f}  {np.mean(r_scores):.4f}")
    print(f"NDCG@{k}         {n_scores[0]:.4f}  {n_scores[1]:.4f}  {n_scores[2]:.4f}  {np.mean(n_scores):.4f}")
    print()

mrr_score = mrr([q["retrieved"] for q in queries], [q["relevant"] for q in queries])
print(f"MRR             : {mrr_score:.4f}")


In [ ]:
import numpy as np

# ============================================================
# FAITHFULNESS -- mesure si la reponse est grounded
# ============================================================
def faithfulness_score(answer_sentences, context):
    """
    Pour chaque phrase de la reponse:
    - est-elle supportee par le contexte ?
    Approximation : overlap de tokens
    """
    def tokens(text):
        return set(text.lower().split())

    context_tokens = tokens(context)
    total, supported = 0, 0
    for sentence in answer_sentences:
        s_tokens = tokens(sentence)
        overlap = len(s_tokens & context_tokens) / len(s_tokens) if s_tokens else 0
        if overlap > 0.5:  # seuil approximatif
            supported += 1
        total += 1
    return supported / total if total > 0 else 0.0

def answer_relevancy_score(answer, question):
    """
    Mesure si la reponse repond a la question (approx)
    Meilleur avec un LLM judge en pratique
    """
    q_words = set(question.lower().split()) - {"quel", "quelle", "le", "la", "de", "est", "?", "en"}
    a_words  = set(answer.lower().split())
    return len(q_words & a_words) / len(q_words) if q_words else 0.0

# Simulation d evaluation RAG
examples = [
    {
        "question": "Quel est le produit net bancaire de BNP en 2024 ?",
        "context":  "En 2024, BNP Paribas a realise un produit net bancaire de 47 milliards d euros. Ce resultat depasse les attentes des analystes.",
        "answer":   "Le produit net bancaire de BNP Paribas en 2024 est de 47 milliards d euros.",
        "reference":"47 milliards d euros",
    },
    {
        "question": "Combien emploie BNP Paribas ?",
        "context":  "BNP Paribas est presente dans 68 pays avec plus de 190 000 collaborateurs.",
        "answer":   "BNP Paribas emploie environ 200 000 personnes dans le monde.",  # hallucination!
        "reference":"190 000 personnes",
    },
    {
        "question": "Comment BNP detecte-t-elle la fraude ?",
        "context":  "BNP utilise des algorithmes de machine learning pour analyser les transactions en temps reel.",
        "answer":   "BNP Paribas utilise le machine learning pour detecter la fraude en analysant les transactions.",
        "reference":"machine learning, temps reel",
    },
]

print("=== Evaluation RAG : Faithfulness + Relevancy ===")
print(f"{'Question':40s}  {'Faith.':>8}  {'Relev.':>8}  {'Issue'}")
for ex in examples:
    sents = [s.strip() for s in ex["answer"].split(".") if s.strip()]
    faith = faithfulness_score(sents, ex["context"])
    relev = answer_relevancy_score(ex["answer"], ex["question"])
    issue = "HALLUCINATION?" if faith < 0.5 else "OK"
    print(f"  {ex['question'][:38]:40s}  {faith:>8.3f}  {relev:>8.3f}  {issue}")

print("\n=== Resume des metriques RAGAS ===")
metrics = [
    ("Faithfulness",       "La reponse est-elle grounded dans le contexte ?",       "0 = hallucination, 1 = 100% grounded"),
    ("Answer Relevancy",   "La reponse repond-elle vraiment a la question ?",        "Score de pertinence [0, 1]"),
    ("Context Precision",  "Parmi les chunks recup., combien sont vraiment utiles ?","Evite le bruit dans le contexte"),
    ("Context Recall",     "Le contexte couvre-t-il toutes les infos necessaires ?", "Evite les reponses incompletes"),
]
for m, desc, note in metrics:
    print(f"  {m:22s}: {desc}")
    print(f"  {'':22s}  -> {note}")
    print()


---
## 8. Questions d’entretien <a name='questions'></a>

**Q : Expliquer HNSW en termes simples.**
> C'est un graphe hierarchique ou les noeuds du bas (couche 0) sont tous les vecteurs avec des connexions locales, et les couches superieures sont des sous-ensembles "de long portee". La recherche commence en haut et descend progressivement en raffinant les candidats. Complexite O(log N) vs O(N) pour le brute force.

**Q : Quand utiliser BM25 vs embeddings ?**
> BM25 est superieur pour les termes exacts, acronymes, noms propres, requetes courtes. Les embeddings gagnent sur les synonymes, paraphrases, langues croisees. En production, utiliser les deux (recherche hybride + RRF).

**Q : Expliquer la difference entre bi-encoder et cross-encoder.**
> Bi-encoder encode query et document separement => dot product => rapide mais perd les interactions. Cross-encoder encode la paire (query, document) ensemble => voit les interactions fines => plus precis mais N fois plus lent. Solution standard : bi-encoder pour le recall (top-100), cross-encoder pour la precision (top-5).

**Q : Qu'est-ce que la faithfulness dans le contexte RAG ?**
> Mesure si chaque affirmation de la reponse est supportee par le contexte recupere. Score de 0 (hallucination totale) a 1 (100% grounded). A distinguer de l'answer relevancy (la reponse repond-elle a la question ?).

**Q : Quels sont les problemes courants en RAG et leurs solutions ?**
> (1) Chunks trop petits => manque de contexte. Solution : augmenter chunk size, parent document retrieval. (2) Hallucination => LLM invente des infos pas dans le contexte. Solution : instruction stricte "ne reponds qu'avec le contexte fourni", RAGAS faithfulness. (3) Lost in the middle => les LLMs oublient les infos au milieu du contexte. Solution : reordering, hierarchical summarization.

**Q : Qu'est-ce que le RRF et pourquoi l'utiliser ?**
> Reciprocal Rank Fusion : `score = sum(1/(k+rank))`. Fusionne plusieurs classements sans se soucier de l'echelle des scores. Robuste, simple, souvent superieur a la combinaison lineaire. Standard pour la recherche hybride BM25 + dense.

**Q : Comment choisir la taille des chunks ?**
> Chunk size ~512 tokens est un bon point de depart. Petits chunks (128-256) = meilleure precision du retrieval mais moins de contexte. Grands chunks (512-1024) = plus de contexte mais plus de bruit. Regle : la reponse a la question doit tenir dans un seul chunk.
